# Lektion 0: Arbeitsumgebung und Auffrischung

Übungsnotebook zu Lektion 0 · Hilfsmittel: Skript Lektion 0

Sie arbeiten als Datenanalyst bei der **Nordlicht Versand GmbH**. In diesem Notebook frischen Sie JOINs und
Gruppierungen auf. Das Datenmodell finden Sie im Skript in Abschnitt 3.

**Hinweis zum Umsatz:** Umsatz einer Position = `menge * einzelpreis`. Stornierte Bestellungen zählen nicht.
Den Rabatt lassen wir vorerst weg.

> **Lösungsnotebook** – nur für die Lehrkraft

## Vorbereitung

Führen Sie zuerst diese Zelle aus (Umschalt + Enter). Danach steht Ihnen `sql(""" ... """)` für Abfragen zur Verfügung.

In [ ]:
import sqlite3
import pandas as pd

pd.set_option("display.max_rows", 100)
con = sqlite3.connect("../daten/shop_oltp.db")

def sql(abfrage):
    """Führt eine SELECT-Abfrage aus und zeigt das Ergebnis als Tabelle."""
    try:
        return pd.read_sql_query(abfrage, con)
    except Exception as e:
        print("Fehler:", str(e).split("': ")[-1])

def ausfuehren(befehl):
    """Führt Befehle ohne Ergebnistabelle aus (CREATE, INSERT, DROP ...)."""
    con.executescript(befehl)
    con.commit()
    print("ausgeführt")

---
### Aufgabe 1: Die Datenbank erkunden

Führen Sie die erste Zelle aus. Sie zeigt alle Tabellen der Datenbank.

Schreiben Sie danach eine Abfrage: **Wie viele Bestellungen gibt es je Status?** Sortieren Sie absteigend nach Anzahl.

In [ ]:
sql("""
SELECT name FROM sqlite_master
WHERE type = 'table'
""")

In [ ]:
sql("""
SELECT status, COUNT(*) AS anzahl
FROM bestellung
GROUP BY status
ORDER BY anzahl DESC
""")

---
### Aufgabe 2: Kunden einer Region

Der Außendienst Süd möchte eine Liste aller Kunden aus der **Region Süd**: Vorname, Nachname und Ort.
Sortieren Sie nach Ort und dann nach Nachname.

In [ ]:
sql("""
SELECT k.vorname, k.nachname, o.name AS ort
FROM kunde k
JOIN ort o    ON o.ort_id = k.ort_id
JOIN region r ON r.region_id = o.region_id
WHERE r.name = 'Süd'
ORDER BY o.name, k.nachname
""")

**Für die Besprechung:** Drei Tabellen, zwei JOINs. Die Region steht nicht beim Kunden, sondern zwei Tabellen weiter.

---
### Aufgabe 3: Eine Bestellung im Detail

Ein Kunde ruft wegen **Bestellung 10** an. Zeigen Sie alle Positionen dieser Bestellung mit
Positionsnummer, Artikelbezeichnung, Kategoriename, Menge, Einzelpreis und dem Betrag (`menge * einzelpreis`).

In [ ]:
sql("""
SELECT p.position_nr, a.bezeichnung, k.name AS kategorie,
       p.menge, p.einzelpreis, p.menge * p.einzelpreis AS betrag
FROM bestellposition p
JOIN artikel a   ON a.artikel_id = p.artikel_id
JOIN kategorie k ON k.kategorie_id = a.kategorie_id
WHERE p.bestellung_id = 10
ORDER BY p.position_nr
""")

---
### Aufgabe 4: Ladenhüter finden

Der Einkauf fragt: **Welche Artikel wurden noch nie verkauft?** Lösen Sie die Aufgabe mit einem LEFT JOIN.

In [ ]:
sql("""
SELECT a.artikel_id, a.bezeichnung
FROM artikel a
LEFT JOIN bestellposition p ON p.artikel_id = a.artikel_id
WHERE p.artikel_id IS NULL
""")

**Für die Besprechung:** Ergebnis: Monitorarm doppelt und Whiteboard 120x90. Typisches Muster „noch nie …“ = LEFT JOIN + IS NULL. In Lektion 3 lösen wir dasselbe mit NOT EXISTS.

---
### Aufgabe 5: INNER JOIN oder LEFT JOIN?

Lassen Sie sich alle Artikel mit ihrem Kategorienamen anzeigen, einmal mit INNER JOIN und einmal mit LEFT JOIN.

a) Wie viele Zeilen liefert jede Variante?
b) Erklären Sie den Unterschied.

In [ ]:
sql("""
SELECT a.bezeichnung, k.name AS kategorie
FROM artikel a
INNER JOIN kategorie k ON k.kategorie_id = a.kategorie_id
""")

In [ ]:
sql("""
SELECT a.bezeichnung, k.name AS kategorie
FROM artikel a
LEFT JOIN kategorie k ON k.kategorie_id = a.kategorie_id
""")

**Musterlösung:**

a) INNER JOIN: 23 Zeilen, LEFT JOIN: 24 Zeilen.

b) Der Artikel „Restposten Kabelset“ hat keine Kategorie (`kategorie_id` ist NULL). Der INNER JOIN findet keinen Partner
und lässt ihn weg. Der LEFT JOIN behält ihn und zeigt als Kategorie NULL (im Notebook als None oder NaN angezeigt).

---
### Aufgabe 6: Wer ist wem unterstellt?

a) Zeigen Sie jeden Mitarbeiter mit seiner Abteilung und dem Namen seines Vorgesetzten. Auch die Geschäftsführung soll erscheinen.

b) Wie viele direkte Mitarbeiter hat jeder Vorgesetzte?

In [ ]:
sql("""
SELECT m.vorname || ' ' || m.nachname AS mitarbeiter, m.abteilung,
       v.vorname || ' ' || v.nachname AS vorgesetzter
FROM mitarbeiter m
LEFT JOIN mitarbeiter v ON v.mitarbeiter_id = m.vorgesetzter_id
""")

In [ ]:
sql("""
SELECT v.vorname || ' ' || v.nachname AS vorgesetzter,
       COUNT(*) AS direkte_mitarbeiter
FROM mitarbeiter m
JOIN mitarbeiter v ON v.mitarbeiter_id = m.vorgesetzter_id
GROUP BY v.mitarbeiter_id, vorgesetzter
ORDER BY direkte_mitarbeiter DESC
""")

**Für die Besprechung:** In b) reicht ein INNER JOIN, denn gezählt werden nur Mitarbeiter, die einen Vorgesetzten haben. Gruppiert wird über die ID, damit zwei Personen mit gleichem Namen nicht zusammenfallen.

---
### Aufgabe 7: Die treuesten Kunden

Ermitteln Sie die **fünf Kunden mit den meisten nicht stornierten Bestellungen**. Zeigen Sie Kunden-ID, Vor- und Nachname und die Anzahl.

In [ ]:
sql("""
SELECT k.kunde_id, k.vorname, k.nachname, COUNT(*) AS bestellungen
FROM kunde k
JOIN bestellung b ON b.kunde_id = k.kunde_id
WHERE b.status <> 'storniert'
GROUP BY k.kunde_id, k.vorname, k.nachname
ORDER BY bestellungen DESC
LIMIT 5
""")

**Für die Besprechung:** Bei Gleichstand auf Platz 5 ist das Ergebnis von LIMIT nicht eindeutig. Gut für die Diskussion: Wie würde man alle Gleichplatzierten zeigen? (Antwort in Lektion 6 mit RANK.)

---
### Aufgabe 8: Umsatz je Kategorie

Wie viel Umsatz hat jede Kategorie **im Jahr 2025** gemacht? Sortieren Sie absteigend nach Umsatz und runden Sie auf zwei Nachkommastellen (`ROUND(wert, 2)`).

In [ ]:
sql("""
SELECT k.name AS kategorie,
       ROUND(SUM(p.menge * p.einzelpreis), 2) AS umsatz
FROM bestellposition p
JOIN bestellung b ON b.bestellung_id = p.bestellung_id
JOIN artikel a    ON a.artikel_id    = p.artikel_id
JOIN kategorie k  ON k.kategorie_id  = a.kategorie_id
WHERE b.status <> 'storniert'
  AND b.bestelldatum BETWEEN '2025-01-01' AND '2025-12-31'
GROUP BY k.name
ORDER BY umsatz DESC
""")

**Für die Besprechung:** Der Restposten ohne Kategorie fällt durch den INNER JOIN heraus. Kann man ansprechen: Fehlt dadurch Umsatz? Mit LEFT JOIN erscheint er als Gruppe NULL.

---
### Aufgabe 9: Bestellungen je Bearbeiter

Der Kundenservice möchte wissen, wie viele Bestellungen jeder Mitarbeiter bearbeitet hat.
Bestellungen **ohne** Bearbeiter sollen als eigene Zeile erscheinen.

In [ ]:
sql("""
SELECT m.vorname || ' ' || m.nachname AS bearbeiter,
       COUNT(*) AS bestellungen
FROM bestellung b
LEFT JOIN mitarbeiter m ON m.mitarbeiter_id = b.mitarbeiter_id
GROUP BY m.mitarbeiter_id, bearbeiter
ORDER BY bestellungen DESC
""")

**Für die Besprechung:** Die Bestellung ist hier die linke Tabelle, weil alle Bestellungen gezählt werden sollen. Die Zeile mit leerem Bearbeiter (NaN) sind die 57 Bestellungen ohne Bearbeiter. Die Aufbereitung als Text „ohne Bearbeiter“ folgt in Lektion 5 mit COALESCE.

---
### Aufgabe 10: Knobelaufgabe: Umsatz nach Region und Kategorie

Die Geschäftsführung möchte für 2025 eine Tabelle: **Umsatz je Region und Kategorie**, sortiert nach Region und Umsatz absteigend.

Zählen Sie anschließend: Wie viele JOINs brauchen Sie dafür? Notieren Sie Ihre Antwort, wir kommen in Lektion 8 darauf zurück.

In [ ]:
sql("""
SELECT r.name AS region, ka.name AS kategorie,
       ROUND(SUM(p.menge * p.einzelpreis), 2) AS umsatz
FROM bestellposition p
JOIN bestellung b  ON b.bestellung_id  = p.bestellung_id
JOIN kunde k       ON k.kunde_id       = b.kunde_id
JOIN ort o         ON o.ort_id         = k.ort_id
JOIN region r      ON r.region_id      = o.region_id
JOIN artikel a     ON a.artikel_id     = p.artikel_id
JOIN kategorie ka  ON ka.kategorie_id  = a.kategorie_id
WHERE b.status <> 'storniert'
  AND b.bestelldatum BETWEEN '2025-01-01' AND '2025-12-31'
GROUP BY r.name, ka.name
ORDER BY r.name, umsatz DESC
""")

**Musterlösung:**

Sechs JOINs über sieben Tabellen. Im Sternschema (Lektion 8) reichen für dieselbe Frage drei JOINs: Faktentabelle mit Kunden-, Artikel- und Datumsdimension.

**Für die Besprechung:** Diese Aufgabe ist die Brücke zum Sternschema. Lassen Sie die Teilnehmenden den Weg durch das Schema an der Tafel einzeichnen.